# 06 — Évaluation de MedGemma sur les erreurs sélectionnées

## Objectif

Ce notebook évalue **MedGemma 1.5 4B IT** sur les questions contenues dans `error_selected.parquet`.

Le protocole reprend celui utilisé pour les autres LLM :

- chargement du projet et des fonctions du dossier `src` ;
- sélection d'une seule ligne par `sample_id` ;
- préparation des prompts avec le même `prompt_v3` ;
- génération des réponses par MedGemma ;
- extraction de la lettre prédite, de la justification et de la confiance ;
- comparaison avec la réponse de référence ;
- sauvegarde progressive des résultats au format Parquet.

> MedGemma est exécuté localement sur le GPU de Google Colab. Afin de limiter l'utilisation de la mémoire GPU, le modèle est chargé en **quantification 4-bit NF4**.


## 1. Préparation de l'environnement

Avant l'exécution, activer un GPU dans Colab :

**Exécution → Modifier le type d'exécution → GPU**

Les dépendances nécessaires à MedGemma et à la quantification 4-bit sont installées ci-dessous.


In [1]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.3 MB/s eta 0:00:00


In [4]:
import sys
from pathlib import Path

import pandas as pd
import torch


## 2. Chargement du projet GitHub

Le dépôt est cloné uniquement s'il n'est pas déjà présent dans `/content`.

Un `git pull` permet ensuite de récupérer les dernières modifications du dossier `src`.


In [5]:
REPO_URL = "https://github.com/cmanell/Medical-LLM-Hallucination-Detection-System.git"
REPO_NAME = "Medical-LLM-Hallucination-Detection-System"

PROJECT_ROOT = Path("/content") / REPO_NAME

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}

%cd {PROJECT_ROOT}
!git pull

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Projet :", PROJECT_ROOT)
print("Dossier src présent :", (PROJECT_ROOT / "src").exists())


/content/Medical-LLM-Hallucination-Detection-System
Already up to date.
Projet : /content/Medical-LLM-Hallucination-Detection-System
Dossier src présent : True


## 3. Import des fonctions du projet

Les fonctions de préparation, d'inférence et d'exécution du benchmark sont importées directement depuis `src`.

Aucune fonction n'est redéfinie dans ce notebook : toute modification du pipeline doit être effectuée dans les fichiers Python du projet, puis récupérée avec Git.


In [6]:
from src.prompt import prepare_sample
from src.llm_inference import run_single_experiment
from src.experiment_runner import run_experiment_batch


## 4. Authentification Hugging Face

MedGemma est un modèle à accès contrôlé.

Il faut avoir accepté les conditions d'utilisation du modèle sur Hugging Face et avoir enregistré un token dans les **Secrets Colab** sous le nom `medgemma_token`.


In [5]:
from google.colab import userdata
from huggingface_hub import login

medgemma_token = userdata.get("medgemma_token")
login(token=medgemma_token)


## 5. Chargement des questions sélectionnées

Le fichier `error_selected.parquet` contient les questions retenues pour l'analyse complémentaire.

Comme une même question peut apparaître plusieurs fois dans ce fichier, notamment pour différents modèles, une seule ligne est conservée par `sample_id`.


In [6]:
ERROR_SELECTED_PATH = (
    PROJECT_ROOT / "final_evaluation" / "error_selected.parquet"
)

df_errors = pd.read_parquet(ERROR_SELECTED_PATH)

print("Dimensions du fichier :", df_errors.shape)
display(df_errors.head())


Dimensions du fichier : (54, 9)


,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_10195,Neurology,"Cas clinique :\nUn homme de 57 ans, ouvrier ag...",E,A,False,Le tableau clinique évoque fortement un accide...,100.0
1,OpenAI,mcqu_train_10195,Neurology,"Cas clinique :\nUn homme de 57 ans, ouvrier ag...",E,A,False,Le tableau évoque un accident vasculaire céréb...,98.0
2,Gemini,mcqu_train_14528,Hepato-Gastroenterology,"Question :\nUn ictère progressif avec prurit, ...",B,D,False,"L'association d'un ictère nu (progressif, sans...",100.0
3,OpenAI,mcqu_train_14528,Hepato-Gastroenterology,"Question :\nUn ictère progressif avec prurit, ...",B,D,False,"Un ictère progressif, indolore, avec prurit, s...",97.0
4,Gemini,mcqu_train_19766,Occupational Medicine,"Question :\nParmi les propositions suivantes, ...",D,C,False,Le certificat médical final de guérison attest...,100.0


In [7]:
df_errors_unique = (
    df_errors
    .drop_duplicates(subset="sample_id")
    .reset_index(drop=True)
)

print("Nombre de questions uniques :", len(df_errors_unique))


Nombre de questions uniques : 27


## 6. Préparation des prompts

Les questions uniques sont converties dans le format attendu par le pipeline d'expérimentation.

La fonction `prepare_sample()` utilise le prompt sélectionné dans le projet et prépare notamment :

- `sample_id` ;
- `prompt_version` ;
- `prompt_text` ;
- `reference_letter` ;
- `medical_subject`.


In [8]:
df_prepared_sample = prepare_sample(df_errors_unique)

print("Nombre d'expériences préparées :", len(df_prepared_sample))
display(df_prepared_sample.head())


Nombre d'expériences préparées : 27


,sample_id,prompt_version,prompt_text,reference_letter,medical_subject
0,mcqu_train_10195,prompt_v3,Vous devez répondre à une question médicale à ...,E,Neurology
1,mcqu_train_14528,prompt_v3,Vous devez répondre à une question médicale à ...,B,Hepato-Gastroenterology
2,mcqu_train_19766,prompt_v3,Vous devez répondre à une question médicale à ...,D,Occupational Medicine
3,mcqu_train_19916,prompt_v3,Vous devez répondre à une question médicale à ...,A,Pharmacology
4,mcqu_train_20638,prompt_v3,Vous devez répondre à une question médicale à ...,D,Psychiatry


## 7. Chargement de MedGemma

Modèle utilisé : **`google/medgemma-1.5-4b-it`**

Signification du nom :

- **Google** : organisation ayant publié le modèle ;
- **MedGemma** : famille de modèles Gemma adaptée au domaine médical ;
- **1.5** : version du modèle ;
- **4B** : environ 4 milliards de paramètres ;
- **IT** : *Instruction Tuned*, c'est-à-dire optimisé pour suivre des instructions.

Le modèle est chargé en **4-bit NF4 avec double quantification** afin de réduire son empreinte mémoire sur le GPU Colab.


In [9]:
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

MODEL_ID = "google/medgemma-1.5-4b-it"

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU CUDA détecté. Active un GPU dans le runtime Colab."
    )

print("GPU :", torch.cuda.get_device_name(0))

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

medgemma_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)

medgemma_model.eval()

print(
    "Empreinte mémoire du modèle :",
    round(medgemma_model.get_memory_footprint() / 1024**3, 2),
    "Go",
)


GPU : Tesla T4


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Empreinte mémoire du modèle : 2.95 Go


## 8. Configuration de la sauvegarde

Les résultats sont enregistrés dans Google Drive afin d'éviter leur perte lors de la fermeture du runtime Colab.

Le fichier Parquet est mis à jour progressivement par `run_experiment_batch()`. Si l'exécution est interrompue, les `sample_id` déjà présents dans le fichier peuvent être ignorés lors de la reprise.


In [8]:
from google.colab import drive

drive.mount("/content/drive")

RESULTS_DIR = Path(
    "/content/drive/MyDrive/data_analytics"
    "Medical-LLM-Hallucination-Detection-System/"
    "data/results"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MEDGEMMA_BENCHMARK_PATH = (
    RESULTS_DIR / "medgemma_results.parquet"
)

print("Résultats :", MEDGEMMA_BENCHMARK_PATH)


Mounted at /content/drive
Résultats : /content/drive/MyDrive/data_analyticsMedical-LLM-Hallucination-Detection-System/data/results/medgemma_results.parquet


## 9. Configuration du backend MedGemma

Contrairement à OpenAI ou Gemini, MedGemma n'utilise pas ici un client API.

Le backend transmis au pipeline contient directement :

- le modèle chargé sur le GPU ;
- le processor utilisé pour formater et décoder les entrées et sorties.


In [11]:
LLM_NAME = "medgemma"
MODEL_NAME = MODEL_ID

medgemma_backend = {
    "model": medgemma_model,
    "processor": processor,
}


## 10. Exécution du benchmark

Chaque question préparée est envoyée à MedGemma.

Les résultats sont sauvegardés progressivement dans `medgemma_results.parquet`. Aucun délai artificiel n'est ajouté entre deux générations car le modèle est exécuté localement et non via une API distante.


In [12]:
df_medgemma_benchmark = run_experiment_batch(
    experiments=df_prepared_sample,
    llm_name=LLM_NAME,
    runner=run_single_experiment,
    model_name=MODEL_NAME,
    llm_client=medgemma_backend,
    output_path=MEDGEMMA_BENCHMARK_PATH,
    pause_seconds=0,
)


Déjà terminées : 0
À exécuter : 27
[1/27] mcqu_train_10195
Prédiction : A | Référence : E | Correcte : False | Erreur : None
[2/27] mcqu_train_14528
Prédiction : A | Référence : B | Correcte : False | Erreur : None
[3/27] mcqu_train_19766
Prédiction : E | Référence : D | Correcte : False | Erreur : None
[4/27] mcqu_train_19916
Prédiction : A | Référence : A | Correcte : True | Erreur : None
[5/27] mcqu_train_20638
Prédiction : A | Référence : D | Correcte : False | Erreur : None
[6/27] mcqu_train_2096
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[7/27] mcqu_train_21179
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[8/27] mcqu_train_22488
Prédiction : E | Référence : C | Correcte : False | Erreur : None
[9/27] mcqu_train_22966
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[10/27] mcqu_train_23113
Prédiction : A | Référence : A | Correcte : True | Erreur : None
[11/27] mcqu_train_23363
Prédiction : C | Référence : D | Correcte : F

## 11. Vérification des résultats

Cette dernière cellule permet de contrôler rapidement :

- le nombre de réponses générées ;
- la proportion de formats de réponse valides ;
- la proportion de réponses correctes parmi les générations effectivement obtenues.


In [13]:
print("Nombre de résultats :", len(df_medgemma_benchmark))

print(
    "Formats valides :",
    df_medgemma_benchmark["response_format_valid"].mean(),
)
print(
    "Exactitude :",
    df_medgemma_benchmark["is_correct"].dropna().mean(),
)

display(
    df_medgemma_benchmark[
        [
            "sample_id",
            "predicted_letter",
            "reference_letter",
            "is_correct",
            "declared_confidence",
            "response_format_valid",
            "generation_error",
        ]
    ]
)


Nombre de résultats : 27
Formats valides : 0.8888888888888888
Exactitude : 0.37037037037037035


,sample_id,predicted_letter,reference_letter,is_correct,declared_confidence,response_format_valid,generation_error
0,mcqu_train_10195,A,E,False,95.0,True,None
1,mcqu_train_14528,A,B,False,95.0,True,None
2,mcqu_train_19766,E,D,False,95.0,True,None
3,mcqu_train_19916,A,A,True,95.0,True,None
4,mcqu_train_20638,A,D,False,NaN,False,None
5,mcqu_train_2096,C,C,True,NaN,False,None
6,mcqu_train_21179,B,B,True,95.0,True,None
7,mcqu_train_22488,E,C,False,90.0,True,None
8,mcqu_train_22966,B,B,True,95.0,True,None
9,mcqu_train_23113,A,A,True,95.0,True,None


## Enregistrement au format csv

##

In [1]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv

In [9]:
df_medgemma_results = pd.read_parquet(MEDGEMMA_BENCHMARK_PATH)

In [11]:
display(df_medgemma_results.head())

,sample_id,model_name,model_version,prompt_version,prompt_text,temperature,raw_response,predicted_letter,generated_justification,declared_confidence,response_format_valid,reference_letter,is_correct,latency_seconds,generation_error
0,mcqu_train_10195,medgemma,google/medgemma-1.5-4b-it,prompt_v3,Vous devez répondre à une question médicale à ...,None,Réponse : A\nJustification : Le tableau cliniq...,A,Le tableau clinique présente des signes d'une ...,95.0,True,E,False,16.741528,None
1,mcqu_train_14528,medgemma,google/medgemma-1.5-4b-it,prompt_v3,Vous devez répondre à une question médicale à ...,None,Réponse : A\nJustification : L'ictère progress...,A,"L'ictère progressif avec prurit, sans douleur,...",95.0,True,B,False,8.833743,None
2,mcqu_train_19766,medgemma,google/medgemma-1.5-4b-it,prompt_v3,Vous devez répondre à une question médicale à ...,None,Réponse : E\nJustification : Le certificat méd...,E,Le certificat médical final de guérison est dé...,95.0,True,D,False,9.016918,None
3,mcqu_train_19916,medgemma,google/medgemma-1.5-4b-it,prompt_v3,Vous devez répondre à une question médicale à ...,None,Réponse : A\nJustification : L'association d'u...,A,L'association d'un anti-inflammatoire non stér...,95.0,True,A,True,13.743702,None
4,mcqu_train_20638,medgemma,google/medgemma-1.5-4b-it,prompt_v3,Vous devez répondre à une question médicale à ...,None,"Réponse : A\nJustification : L'état fébrile, l...",A,"L'état fébrile, les tremblements, l'halètement...",NaN,False,D,False,41.264743,None


In [12]:
df_medgemma_results.to_csv(RESULTS_DIR / "error_selected.csv", index=False,  encoding="utf-8-sig")
